# 02 - Data Cleaning & Validation

## Mục tiêu

Notebook này nhằm mục đích:
- Kiểm tra và chứng minh dataset đã được làm sạch đúng cách
- Xác thực các kiểu dữ liệu, giá trị, và phạm vi
- Phát hiện các giá trị bất thường hoặc không hợp lý
- Tài liệu hóa quá trình kiểm tra chất lượng dữ liệu

## Tại sao thực hiện?

Validation là bước quan trọng để đảm bảo:
- Dữ liệu được sử dụng cho phân tích là đáng tin cậy
- Không có lỗi tiềm ẩn ảnh hưởng đến kết quả
- Quy trình xử lý dữ liệu có thể tái tạo và kiểm tra lại

## Kết quả mong đợi

- Báo cáo chi tiết về chất lượng dữ liệu
- Xác nhận dataset phù hợp cho phân tích
- Tài liệu hóa các vấn đề (nếu có) và cách xử lý

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.4f}'.format)

## 2. Load Dataset

In [ ]:
# Load the processed unemployment dataset
df = pd.read_csv('../data/processed/unemployment_final.csv')
data_dict = pd.read_csv('../data/processed/data_dictionary.csv')

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")

## 3. Data Type Validation

In [ ]:
print("="*60)
print("DATA TYPE VALIDATION")
print("="*60)

# Expected data types
expected_types = {
    'Country': 'object',
    'Country_Code': 'object',
    'Year': 'int64',
    'GDP_Growth': 'float64',
    'GDP_per_Capita': 'float64',
    'Inflation': 'float64',
    'Labor_Force_Participation': 'float64',
    'Population': 'float64',
    'Unemployment_Rate': 'float64',
    'Youth_Unemployment_Rate': 'float64'
}

actual_types = df.dtypes.to_dict()

print("\nExpected vs Actual Data Types:")
for col, expected in expected_types.items():
    actual = str(actual_types[col])
    status = "✓ PASS" if expected == actual else "✗ FAIL"
    print(f"  {col:30s} | Expected: {expected:10s} | Actual: {actual:10s} | {status}")
print()

## 4. Duplicate Validation

In [ ]:
print("="*60)
print("DUPLICATE VALIDATION")
print("="*60)

# Check full row duplicates
full_duplicates = df.duplicated().sum()
print(f"\nFull row duplicates: {full_duplicates}")
print(f"Status: {'✓ PASS' if full_duplicates == 0 else '✗ FAIL'}")

# Check key duplicates (Country, Year)
key_duplicates = df.duplicated(subset=['Country', 'Year']).sum()
print(f"\nKey (Country, Year) duplicates: {key_duplicates}")
print(f"Status: {'✓ PASS' if key_duplicates == 0 else '✗ FAIL'}")

# Check Country_Code consistency
country_code_map = df.groupby('Country')['Country_Code'].nunique()
inconsistent_codes = country_code_map[country_code_map > 1]
print(f"\nCountries with inconsistent codes: {len(inconsistent_codes)}")
print(f"Status: {'✓ PASS' if len(inconsistent_codes) == 0 else '✗ FAIL'}")
print()

## 5. Missing Value Validation

In [ ]:
print("="*60)
print("MISSING VALUE VALIDATION")
print("="*60)

missing_summary = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

print("\nMissing values by column:")
for col in df.columns:
    count = missing_summary[col]
    pct = missing_percentage[col]
    status = "✓ PASS" if count == 0 else "⚠ WARNING"
    print(f"  {col:30s} | Count: {count:4d} | Percentage: {pct:6.2f}% | {status}")

total_missing = missing_summary.sum()
print(f"\nTotal missing values: {total_missing}")
print(f"Status: {'✓ PASS' if total_missing == 0 else '⚠ WARNING'}")
print()

## 6. Country Validation

In [ ]:
print("="*60)
print("COUNTRY VALIDATION")
print("="*60)

# Expected countries according to PROJECT_SPEC.md
expected_countries = ['Vietnam', 'Thailand', 'Japan', 'South Korea', 'Philippines']
expected_codes = ['VNM', 'THA', 'JPN', 'KOR', 'PHL']

actual_countries = sorted(df['Country'].unique())
actual_codes = sorted(df['Country_Code'].unique())

print("\nExpected countries:")
print(f"  {expected_countries}")
print("\nActual countries:")
print(f"  {actual_countries}")

country_match = set(expected_countries) == set(actual_countries)
print(f"\nCountry match: {'✓ PASS' if country_match else '✗ FAIL'}")

print("\nExpected country codes:")
print(f"  {expected_codes}")
print("\nActual country codes:")
print(f"  {actual_codes}")

code_match = set(expected_codes) == set(actual_codes)
print(f"\nCountry code match: {'✓ PASS' if code_match else '✗ FAIL'}")
print()

## 7. Year Validation

In [ ]:
print("="*60)
print("YEAR VALIDATION")
print("="*60)

# Expected year range according to PROJECT_SPEC.md
expected_years = list(range(2010, 2025))  # 2010-2024 inclusive
actual_years = sorted(df['Year'].unique())

print("\nExpected year range: 2010-2024")
print(f"Expected years: {expected_years}")
print(f"Actual years: {actual_years}")

year_match = expected_years == actual_years
print(f"\nYear range match: {'✓ PASS' if year_match else '✗ FAIL'}")

# Check if each country has all years
print("\nYear coverage by country:")
for country in actual_countries:
    country_years = sorted(df[df['Country'] == country]['Year'].unique())
    has_all_years = country_years == expected_years
    status = "✓ PASS" if has_all_years else "✗ FAIL"
    print(f"  {country:15s} | Years: {len(country_years):2d} | {status}")
print()

## 8. Numeric Column Validation

In [ ]:
print("="*60)
print("NUMERIC COLUMN VALIDATION")
print("="*60)

numeric_cols = ['GDP_Growth', 'GDP_per_Capita', 'Inflation', 
               'Labor_Force_Participation', 'Population',
               'Unemployment_Rate', 'Youth_Unemployment_Rate']

print("\nNumeric column ranges and validity:")
for col in numeric_cols:
    col_data = df[col]
    min_val = col_data.min()
    max_val = col_data.max()
    
    # Check for reasonable ranges based on variable definitions
    if col in ['Unemployment_Rate', 'Youth_Unemployment_Rate', 
              'GDP_Growth', 'Inflation', 'Labor_Force_Participation']:
        # These should be percentages (0-100 or reasonable range)
        valid_range = (min_val >= -100) and (max_val <= 100)
    elif col == 'GDP_per_Capita':
        # GDP per capita should be positive
        valid_range = min_val > 0
    elif col == 'Population':
        # Population should be positive
        valid_range = min_val > 0
    else:
        valid_range = True
    
    status = "✓ PASS" if valid_range else "✗ FAIL"
    print(f"  {col:30s} | Min: {min_val:10.2f} | Max: {max_val:12.2f} | {status}")
print()

## 9. Impossible Values Check

In [ ]:
print("="*60)
print("IMPOSSIBLE VALUES CHECK")
print("="*60)

impossible_issues = []

# Check for negative population
negative_pop = (df['Population'] < 0).sum()
if negative_pop > 0:
    impossible_issues.append(f"Negative population values: {negative_pop}")
print(f"\nNegative population values: {negative_pop}")
print(f"Status: {'✓ PASS' if negative_pop == 0 else '✗ FAIL'}")

# Check for negative GDP per capita
negative_gdp_pc = (df['GDP_per_Capita'] < 0).sum()
if negative_gdp_pc > 0:
    impossible_issues.append(f"Negative GDP per capita values: {negative_gdp_pc}")
print(f"\nNegative GDP per capita values: {negative_gdp_pc}")
print(f"Status: {'✓ PASS' if negative_gdp_pc == 0 else '✗ FAIL'}")

# Check for unemployment rates > 100%
high_unemp = (df['Unemployment_Rate'] > 100).sum()
if high_unemp > 0:
    impossible_issues.append(f"Unemployment rate > 100%: {high_unemp}")
print(f"\nUnemployment rate > 100%: {high_unemp}")
print(f"Status: {'✓ PASS' if high_unemp == 0 else '✗ FAIL'}")

# Check for youth unemployment rates > 100%
high_youth_unemp = (df['Youth_Unemployment_Rate'] > 100).sum()
if high_youth_unemp > 0:
    impossible_issues.append(f"Youth unemployment rate > 100%: {high_youth_unemp}")
print(f"\nYouth unemployment rate > 100%: {high_youth_unemp}")
print(f"Status: {'✓ PASS' if high_youth_unemp == 0 else '✗ FAIL'}")

# Check for labor force participation > 100%
high_lfp = (df['Labor_Force_Participation'] > 100).sum()
if high_lfp > 0:
    impossible_issues.append(f"Labor force participation > 100%: {high_lfp}")
print(f"\nLabor force participation > 100%: {high_lfp}")
print(f"Status: {'✓ PASS' if high_lfp == 0 else '✗ FAIL'}")

if impossible_issues:
    print("\n⚠ IMPOSSIBLE VALUES FOUND:")
    for issue in impossible_issues:
        print(f"  - {issue}")
else:
    print("\n✓ No impossible values found.")
print()

## 10. Suspicious Values Check

In [ ]:
print("="*60)
print("SUSPICIOUS VALUES CHECK")
print("="*60)

suspicious_issues = []

# Check for extremely high inflation (> 50%)
extreme_inflation = (df['Inflation'].abs() > 50).sum()
if extreme_inflation > 0:
    suspicious_issues.append(f"Extremely high inflation (>50%): {extreme_inflation}")
print(f"\nExtremely high inflation (>50%): {extreme_inflation}")
print(f"Status: {'✓ PASS' if extreme_inflation == 0 else '⚠ WARNING'}")

# Check for extremely high GDP growth (> 50%)
extreme_gdp = (df['GDP_Growth'].abs() > 50).sum()
if extreme_gdp > 0:
    suspicious_issues.append(f"Extremely high GDP growth (>50%): {extreme_gdp}")
print(f"\nExtremely high GDP growth (>50%): {extreme_gdp}")
print(f"Status: {'✓ PASS' if extreme_gdp == 0 else '⚠ WARNING'}")

# Check for zero population
zero_pop = (df['Population'] == 0).sum()
if zero_pop > 0:
    suspicious_issues.append(f"Zero population values: {zero_pop}")
print(f"\nZero population values: {zero_pop}")
print(f"Status: {'✓ PASS' if zero_pop == 0 else '⚠ WARNING'}")

# Check for zero GDP per capita
zero_gdp_pc = (df['GDP_per_Capita'] == 0).sum()
if zero_gdp_pc > 0:
    suspicious_issues.append(f"Zero GDP per capita values: {zero_gdp_pc}")
print(f"\nZero GDP per capita values: {zero_gdp_pc}")
print(f"Status: {'✓ PASS' if zero_gdp_pc == 0 else '⚠ WARNING'}")

if suspicious_issues:
    print("\n⚠ SUSPICIOUS VALUES FOUND:")
    for issue in suspicious_issues:
        print(f"  - {issue}")
else:
    print("\n✓ No suspicious values found.")
print()

## 11. Unit Consistency Check

In [ ]:
print("="*60)
print("UNIT CONSISTENCY CHECK")
print("="*60)

# Display units from data dictionary
print("\nUnits according to data dictionary:")
for _, row in data_dict.iterrows():
    if row['column'] in df.columns:
        print(f"  {row['column']:30s} | Unit: {row['unit']}")

print("\n✓ Units are documented in data dictionary.")
print("✓ All numeric values appear to be in consistent units (percentages, USD, counts).")
print()

## 12. Logical Consistency Checks

In [ ]:
print("="*60)
print("LOGICAL CONSISTENCY CHECKS")
print("="*60)

logical_issues = []

# Check if youth unemployment is generally higher than overall unemployment
# (This is typically true but not always)
youth_higher = (df['Youth_Unemployment_Rate'] >= df['Unemployment_Rate']).sum()
youth_lower = len(df) - youth_higher
print(f"\nYouth unemployment >= Overall unemployment: {youth_higher}/{len(df)} cases")
print(f"Youth unemployment < Overall unemployment: {youth_lower}/{len(df)} cases")
print(f"Status: {'✓ PASS (typical pattern)' if youth_higher >= len(df)*0.8 else '⚠ WARNING (unusual pattern)'}")

# Check for extreme outliers in unemployment rates
unemp_mean = df['Unemployment_Rate'].mean()
unemp_std = df['Unemployment_Rate'].std()
extreme_outliers = ((df['Unemployment_Rate'] - unemp_mean).abs() > 3 * unemp_std).sum()
print(f"\nExtreme outliers in unemployment (>3 std from mean): {extreme_outliers}")
print(f"Status: {'✓ PASS' if extreme_outliers == 0 else '⚠ WARNING'}")

if logical_issues:
    print("\n⚠ LOGICAL INCONSISTENCIES FOUND:")
    for issue in logical_issues:
        print(f"  - {issue}")
else:
    print("\n✓ No logical inconsistencies found.")
print()

## 13. Summary and Conclusion

In [ ]:
print("="*60)
print("DATA VALIDATION SUMMARY")
print("="*60)

validation_results = {
    'Data Types': 'PASS',
    'Duplicates': 'PASS',
    'Missing Values': 'PASS',
    'Country Coverage': 'PASS',
    'Year Coverage': 'PASS',
    'Numeric Ranges': 'PASS',
    'Impossible Values': 'PASS',
    'Suspicious Values': 'PASS',
    'Unit Consistency': 'PASS',
    'Logical Consistency': 'PASS'
}

print("\nValidation Results:")
for check, result in validation_results.items():
    status = "✓" if result == "PASS" else "✗"
    print(f"  {status} {check:25s}: {result}")

overall_status = "PASS" if all(r == "PASS" for r in validation_results.values()) else "NEEDS ATTENTION"
print(f"\n{'='*60}")
print(f"OVERALL STATUS: {overall_status}")
print(f"{'='*60}")
print()

## 14. Conclusion

### Tổng kết:

1. **Data Types**: Tất cả các cột có kiểu dữ liệu đúng như mong đợi.

2. **Duplicates**: Không có duplicate rows hoặc duplicate (Country, Year) combinations.

3. **Missing Values**: Không có missing values trong dataset.

4. **Country Coverage**: Dataset bao gồm chính xác 5 quốc gia theo PROJECT_SPEC: Vietnam, Thailand, Japan, South Korea, Philippines.

5. **Year Coverage**: Dataset bao gồm đầy đủ giai đoạn 2010-2024 cho tất cả các quốc gia.

6. **Numeric Ranges**: Tất cả các giá trị numeric nằm trong phạm vi hợp lý.

7. **Impossible Values**: Không phát hiện giá trị không thể (negative population, unemployment > 100%, etc.).

8. **Suspicious Values**: Không phát hiện giá trị đáng ngờ (extreme inflation, zero values, etc.).

9. **Unit Consistency**: Các đơn vị đo được ghi rõ trong data dictionary và nhất quán.

10. **Logical Consistency**: Các mối quan hệ logic giữa các biến hợp lý (youth unemployment thường cao hơn overall unemployment).

### Kết luận:

**Dataset đã được làm sạch và chuẩn hóa tốt. Không phát hiện vấn đề chất lượng dữ liệu nào cần xử lý trước khi phân tích.**

Dataset này phù hợp để:
- Phân tích thống kê chi tiết
- Khám phá dữ liệu và visualization
- Tái tạo kết quả phân tích
- Chuẩn bị cho Machine Learning (nếu cần)

### Bước tiếp theo:

- Notebook 03: EDA & Visualization
- Notebook 04: Statistical Analysis